In [1]:
# Trigram Model (counting)

# Step 1: form trigrams, need to figure out the best way to do this, could be a massive matrix, that seems very inefficient
# I think I will just try it with a massive non-square matrix, rows will be combinations (ab, ac, ad, etc.), columns will be 
# counts. Correct implementation uses Counter which makes a dictionary
from modulefinder import test
import random
from re import S
import torch 

words = open('names.txt', 'r').read().splitlines()

random.Random(42).shuffle(words)

train_end = int(len(words) * 0.8)
validation_end = int(len(words) * 0.9)

train = words[:train_end]
validation = words[train_end : validation_end]
test_set = words[validation_end:]

chars = set(''.join(words))
chars.add('.')

N = torch.zeros(729, 27, dtype=torch.int32)

stoi = {ch : i for i, ch in enumerate(chars)}

for w in train:
    chs = ['.', '.'] + list(w) + ['.']

    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        row = 27 * stoi[ch1] + stoi[ch2]
        col = stoi[ch3]
        N[row, col] += 1
        

In [2]:
len(words)


32033

In [3]:
# Step 2: form probability matrix based on counts
P = (N + 1).float()
P /= P.sum(1, keepdim=True)

In [4]:
# n Trigram Samples
itos = {i : s for s, i in stoi.items()}
g = torch.Generator().manual_seed(2147483647)
n = 5

for i in range(5):

    out = []
    ix1 = ix2 = stoi['.']
    while True:
        row = 27 * ix1 + ix2
        p = P[row]

        ix3 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        
        if ix3 == stoi['.']:
            break
        out.append(itos[ix3])
        ix1, ix2 = ix2, ix3

    print(''.join(out))
    

chakin
ad
ry
bla
karines


In [5]:
# Neural net approach
# Step 1: Form training dataset, onehot encode weights
# Step 2: Loss function
# Step 3: Instantiate random weight matrix
# Step 4: Gradient descent

In [6]:
# Step 1: Training dataset pairs
xs, ys = [], []

for w in train:
    chs = ['.', '.'] + list(w) + ['.']
    
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        xs.append(27 * stoi[ch1] + stoi[ch2])
        ys.append(stoi[ch3])

xs = torch.tensor(xs, dtype=torch.long)
ys = torch.tensor(ys, dtype=torch.long)

In [7]:
# Step 2
W_init = torch.randn((729, 27), generator=g)
W_sgd = W_init.clone().requires_grad_()
W_AdamW = W_init.clone().requires_grad_()

In [8]:
# Different optimizers
import torch.nn.functional as F
from math import inf

lams = [0.01, 0.03, 0.05, 0.10]
lam = 1e-6

eps = 1e-5
learning_rate = 1.0
max_epochs = 1000
w = 0.2


In [9]:
# SGD w/ Momentum
diff = inf
i = 0
u = torch.zeros((729, 27), dtype=torch.float32)
mu = 0.9

while i < max_epochs and diff > eps:
    logits = W_sgd[xs]
    loss = F.cross_entropy(logits, ys) #+ lam * W_sgd.square().sum()    
    W_sgd.grad = None
    loss.backward()
    
    with torch.no_grad():
        u = mu * u + W_sgd.grad
        W_sgd -= learning_rate * W_sgd.grad + learning_rate * mu * u

    i += 1
    diff = W_sgd.grad.norm().item()
    if i % 100 == 0:
        print(loss.item())

2.998887777328491
2.7596335411071777
2.639256477355957
2.5631871223449707
2.509925365447998
2.470322370529175
2.439633846282959
2.4150960445404053
2.395002841949463
2.3782458305358887


In [ ]:
# AdamW
optimizer = torch.optim.Adam([W_AdamW], lr=learning_rate, weight_decay=w)

for i in range(max_epochs):
    optimizer.zero_grad(set_to_none=True)

    loss = F.cross_entropy(W_AdamW[xs], ys)
    loss.backward()
    optimizer.step()

    if i % 100 == 0:
        print(loss.item())


3.7730817794799805
3.2887964248657227
3.2887394428253174
3.288740873336792
3.288740873336792
3.288740873336792
3.288740873336792
3.288740873336792
3.288740873336792


In [11]:
ix1 = ix2 = stoi['.']
out = []

with torch.no_grad():
    while True:
        row = 27 * ix1 + ix2
        probs = torch.softmax(W_sgd[row], dim=-1)
        

        ix3 = torch.multinomial(probs, num_samples=1).item()

        if itos[ix3] == '.':
            break
            
        out.append(itos[ix3])
        ix1 = ix2
        ix2 = ix3

print(''.join(out))

n


In [12]:
ix1 = ix2 = stoi['.']
out = []

with torch.no_grad():
    while True:
        row = 27 * ix1 + ix2
        probs = torch.softmax(W_AdamW[row], dim=-1)

        ix3 = torch.multinomial(probs, num_samples=1).item()

        if itos[ix3] == '.':
            break
            
        out.append(itos[ix3])
        ix1 = ix2
        ix2 = ix3

print(''.join(out))

tieuyxxrbyncipsifll


In [13]:
# Cross validation
def build_dataset(word_list):
    xs, ys = [], []

    for w in word_list:
        chs = ['.', '.'] + list(w) + ['.']

        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            xs.append(27 * stoi[ch1] + stoi[ch2])
            ys.append(stoi[ch3])

    return (
        torch.tensor(xs, dtype=torch.long),
        torch.tensor(ys, dtype=torch.long),
    )

In [14]:
xs_val, ys_val = build_dataset(validation)

with torch.no_grad():
    val_loss = F.cross_entropy(W_sgd[xs_val], ys_val).item()

print("Validation CE:", val_loss)



Validation CE: 2.7497682571411133


In [15]:
def fit(xs, ys, lr, weight_decay, steps=1000):
    W = W_init.detach().clone().requires_grad_()

    optimizer = torch.optim.AdamW(
        [W],
        lr=lr,
        weight_decay=weight_decay,
    )

    for _ in range(steps):
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(W[xs], ys)
        loss.backward()
        optimizer.step()

    return W.detach()

In [16]:
from sklearn.model_selection import KFold
from joblib import Parallel, delayed, parallel_config
from itertools import product
import torch.nn.functional as F

pool = train + validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

fold_data = []

for train_ids, val_ids in cv.split(pool):
    x_train, y_train = build_dataset([pool[i] for i in train_ids])
    x_val, y_val = build_dataset([pool[i] for i in val_ids])

    fold_data.append((x_train, y_train, x_val, y_val))

W0 = W_init.detach().cpu().clone()

In [17]:
def cv_score(lr, wd, fold_data, W0, steps=1000):
    torch.set_num_threads(1)
    fold_losses = []

    for x_train, y_train, x_val, y_val in fold_data:
        W = W0.clone().requires_grad_()
        optimizer = torch.optim.AdamW(
            [W], lr=lr, weight_decay=wd
        )

        for _ in range(steps):
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(W[x_train], y_train)
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            val_loss = F.cross_entropy(W[x_val], y_val).item()

        fold_losses.append(val_loss)

    return sum(fold_losses) / len(fold_losses), lr, wd

In [20]:
learning_rates = [0.5, 0.75, 1.0]
weight_decays = [0.2, 0.3, 0.5]

with parallel_config(
    backend="loky",
    n_jobs=6,
    inner_max_num_threads=1,
):
    results = Parallel(verbose=10)(
        delayed(cv_score)(lr, wd, fold_data, W0)
        for lr, wd in product(learning_rates, weight_decays)
    )

results.sort()  # Lowest average validation loss first
best_loss, best_lr, best_wd = results[0]

print("Best CV loss:", best_loss)
print("Best learning rate:", best_lr)
print("Best weight decay:", best_wd)

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   2 out of   9 | elapsed:  1.6min remaining:  5.6min
[Parallel(n_jobs=6)]: Done   3 out of   9 | elapsed:  1.6min remaining:  3.2min
[Parallel(n_jobs=6)]: Done   4 out of   9 | elapsed:  1.6min remaining:  2.0min
[Parallel(n_jobs=6)]: Done   5 out of   9 | elapsed:  1.6min remaining:  1.3min
[Parallel(n_jobs=6)]: Done   6 out of   9 | elapsed:  1.6min remaining:   49.1s
[Parallel(n_jobs=6)]: Done   7 out of   9 | elapsed:  2.6min remaining:   44.7s


Best CV loss: 2.2347613334655763
Best learning rate: 1.0
Best weight decay: 0.2


[Parallel(n_jobs=6)]: Done   9 out of   9 | elapsed:  3.0min finished
